In [ ]:
import numpy as np
import pandas as pd
from collections import deque
from provenance import Provenance
import algebra
import networkx as nx

t = Provenance()

In [2]:
# --- Data Loading ---
print("LOADING REAL BIBLE DATA (BradyStephenson/bible-data)")

base_url = "https://raw.githubusercontent.com/BradyStephenson/bible-data/main"
df_persons = pd.read_csv(f"{base_url}/BibleData-Person.csv")
df_rels    = pd.read_csv(f"{base_url}/BibleData-PersonRelationship.csv")

print(f"Loaded {len(df_persons)} people and {len(df_rels)} relationship records.")

LOADING REAL BIBLE DATA (BradyStephenson/bible-data)
Loaded 3009 people and 5450 relationship records.


In [3]:
# --- Normalize to Parent → Child direction ---
target_rels = ['father', 'mother', 'son', 'daughter']
df_family = df_rels[df_rels['relationship_type'].str.lower().isin(target_rels)].copy()

mask_parent = df_family['relationship_type'].str.lower().isin(['father', 'mother'])

df_family['parent_id'] = np.where(mask_parent, df_family['person_id_1'], df_family['person_id_2'])
df_family['child_id']  = np.where(mask_parent, df_family['person_id_2'], df_family['person_id_1'])
df_family = df_family.dropna(subset=['parent_id', 'child_id'])

In [4]:
# --- Build adjacency matrix ---
valid_ids = sorted(set(df_family['parent_id']) | set(df_family['child_id']))
id_to_idx = {rid: i for i, rid in enumerate(valid_ids)}
N         = len(valid_ids)

id_to_name = df_persons.set_index('person_id')['person_name']
names      = [id_to_name.get(rid, f"Unknown_{rid}") for rid in valid_ids]

Parent = np.zeros((N, N), dtype=int)
p_idx  = [id_to_idx[p] for p in df_family['parent_id']]
c_idx  = [id_to_idx[c] for c in df_family['child_id']]
Parent[p_idx, c_idx] = 1

print(f"Matrix: {Parent.shape} ({N} nodes)")

Matrix: (1972, 1972) (1972 nodes)


In [5]:
def print_relationships(matrix, title):
    count = int(np.sum(matrix))
    print(f"\n{title}: {count} relationships found")
    return count

print_relationships(Parent, "Direct parent relationships")


Direct parent relationships: 1727 relationships found


1727

In [6]:

from fixpoint import FixpointIterator
from algebra import Abs, Sum

def _f(R, temp):
    J            = t.Join(R, Parent, temp)
    Rn           = t.SmoothMax((J, Parent), temp, axis=0)
    np.fill_diagonal(Rn, 0)
    R_allowed    = t.Residuate(Rn, Parent, temp)
    Rn_corrected = t.SmoothMin((Rn, R_allowed), temp, axis=0)
    return Rn_corrected, Rn

def _energy(new, old, aux):
    Rn = aux
    return Sum(Abs(new - old) ** 2) + Sum(Abs(Rn - new) ** 2)

fp_family = FixpointIterator(f=_f, energy_fn=_energy, state0=Parent.copy(),
                              eps=1e-3, max_iters=100)

print("\nCONVERGENCE ITERATIONS")
fp_family.run()
print(fp_family)



CONVERGENCE ITERATIONS
FixpointIterator(iter=75, energy=0.0000, temp=0.0000)


In [7]:
family_tree = fp_family.state

In [8]:
# --- Verify ---
def verify_transitive_closure(P, A):
    P = (P > 0).astype(int)
    A = (A > 0).astype(int)

    print("[1/3] Checking: Parent ⊆ Ancestor")
    assert np.all((P == 0) | (A == 1)), "Missing a direct parent link in Ancestor"
    print("  ✓ PASS")

    print("[2/3] Checking: Closure property (A @ Parent adds nothing)")
    assert not np.any((A @ P > 0) & (A == 0)), "Closure not reached (A @ Parent found new edges)"
    print("  ✓ PASS")

    print("[3/3] Checking: No cycles (diagonal = 0)")
    cycles = int(np.diag(A).sum())
    if cycles > 0:
        print(f"  ! NOTE: {cycles} self-loops found (data quirks / loops in raw dataset).")
    else:
        print("  ✓ PASS")

print("\nFINAL RESULTS")
initial_count = print_relationships(Parent,      "Direct parent relationships")
final_count   = print_relationships(family_tree, "Complete ancestor relationships")
print(f"Total new links discovered: {final_count - initial_count}")



FINAL RESULTS

Direct parent relationships: 1727 relationships found

Complete ancestor relationships: 33943 relationships found
Total new links discovered: 32216


In [9]:
try:
    verify_transitive_closure(Parent, family_tree)
    print("\n *** ALL CHECKS PASSED ***")
except AssertionError as e:
    print(f"\n --- Verification failed --- : {e}")

[1/3] Checking: Parent ⊆ Ancestor
  ✓ PASS
[2/3] Checking: Closure property (A @ Parent adds nothing)
  ✓ PASS
[3/3] Checking: No cycles (diagonal = 0)
  ✓ PASS

 *** ALL CHECKS PASSED ***


In [10]:
# --- Lineage queries ---
def show_lineage(name_query, top_n=None):
    def bfs_levels(adj_matrix, start_idx):
        dist = {start_idx: 0}
        q = deque([start_idx])
        while q:
            u = q.popleft()
            for v in np.where(adj_matrix[u] == 1)[0]:
                if v not in dist:
                    dist[v] = dist[u] + 1
                    q.append(v)
        return dist

    try:
        idx = next(i for i, n in enumerate(names) if name_query.lower() in n.lower())
    except StopIteration:
        print(f"\nCould not find '{name_query}' in the dataset.")
        return

    name       = names[idx]
    total_desc = int((family_tree[idx] > 0).sum())
    total_anc  = int((family_tree[:, idx] > 0).sum())

    print(f"\n[{name}]")
    print(f"  → Ancestor of {total_desc} people in total.")
    print(f"  ← Descended from {total_anc} people in total.")

    if top_n is None:
        return

    desc_dist = bfs_levels(Parent, idx)
    desc_dist.pop(idx, None)
    sorted_desc = sorted(desc_dist.items(), key=lambda x: (x[1], names[x[0]]))[:top_n]

    anc_dist = bfs_levels(Parent.T, idx)
    anc_dist.pop(idx, None)
    sorted_anc = sorted(anc_dist.items(), key=lambda x: (x[1], names[x[0]]))[:top_n]

    if sorted_desc:
        print(f"Top {len(sorted_desc)} descendants by generation:")
        for node_idx, dist in sorted_desc:
            print(f"    gen {dist:2d}: {names[node_idx]}")
    else:
        print("No descendants in the dataset.")

    if sorted_anc:
        print(f"Top {len(sorted_anc)} ancestors by generation:")
        for node_idx, dist in sorted_anc:
            print(f"    gen {dist:2d}: {names[node_idx]}")
    else:
        print("No ancestors in the dataset.")

In [11]:
print("\nINTERPRETATION (Selected Bible Figures)")
show_lineage("Adam")
show_lineage("Adam", top_n=5)
show_lineage("Abram", top_n=10)


INTERPRETATION (Selected Bible Figures)

[Adam]
  → Ancestor of 821 people in total.
  ← Descended from 0 people in total.

[Adam]
  → Ancestor of 821 people in total.
  ← Descended from 0 people in total.
Top 5 descendants by generation:
    gen  1: Abel
    gen  1: Cain
    gen  1: Seth
    gen  2: Enoch
    gen  2: Enosh
No ancestors in the dataset.

[Abram]
  → Ancestor of 698 people in total.
  ← Descended from 20 people in total.
Top 10 descendants by generation:
    gen  1: Isaac
    gen  1: Ishbak
    gen  1: Ishmael
    gen  1: Jokshan
    gen  1: Medan
    gen  1: Midian
    gen  1: Shuah
    gen  1: Zimran
    gen  2: Abida
    gen  2: Adbeel
Top 10 ancestors by generation:
    gen  1: Terah
    gen  2: Nahor
    gen  3: Serug
    gen  4: Reu
    gen  5: Peleg
    gen  6: Eber
    gen  7: Shelah
    gen  8: Arpachshad
    gen  9: Shem
    gen 10: Noah


In [ ]:
# Find indices for test figures
adam_idx  = next(i for i, n in enumerate(names) if "Adam"  == n)
isaac_idx = next(i for i, n in enumerate(names) if "Isaac" == n)
abram_idx = next(i for i, n in enumerate(names) if "Abram" == n)
noah_idx  = next(i for i, n in enumerate(names) if "Noah"  == n)

print(f"Adam  → idx {adam_idx}  ({names[adam_idx]})")
print(f"Isaac → idx {isaac_idx} ({names[isaac_idx]})")
print(f"Abram → idx {abram_idx} ({names[abram_idx]})")

# Query: is Adam an ancestor of Abram?
result = t.Query(family_tree, src=adam_idx, dst=abram_idx, names=names, relation="ancestor of", isClosed=True)
print("\n--- Adam → Abram ---")
print(result)

Adam  → idx 66  (Adam)
Isaac → idx 815 (Isaac)
Abram → idx 48 (Abram)
[Query] Adam → Abram  relation='ancestor of'  threshold=0.05  temp=0.05
[Query] proof found

--- Adam → Abram ---
{'node': 'Adam ancestor of Abram because:', 'branches': ['Adam (1.000) ancestor of → Seth (1.000) ancestor of → Enosh (1.000) ancestor of → Kenan (1.000) ancestor of → Mahalalel (1.000) ancestor of → Jared (1.000) ancestor of → Enoch (1.000) ancestor of → Methuselah (1.000) ancestor of → Lamech (1.000) ancestor of → Noah (1.000) ancestor of → Shem (1.000) ancestor of → Arpachshad (1.000) ancestor of → Shelah (1.000) ancestor of → Eber (1.000) ancestor of → Peleg (1.000) ancestor of → Reu (1.000) ancestor of → Serug (1.000) ancestor of → Nahor (1.000) ancestor of → Terah (1.000) ancestor of → Abram']}


In [15]:
import matplotlib.pyplot as plt

G_trace = t.GetWitnessGraph(family_tree, adam_idx, abram_idx)

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G_trace, k=0.5, seed=42)

lengths = nx.single_source_shortest_path_length(G_trace, adam_idx)
node_colors = [lengths.get(node, 0) for node in G_trace.nodes()]

nx.draw_networkx_edges(G_trace, pos, width=2, edge_color='red', arrowsize=20)
nx.draw_networkx_nodes(G_trace, pos, node_size=800, node_color=node_colors, cmap=plt.cm.plasma)
node_labels = {idx: names[idx] for idx in G_trace.nodes()}
nx.draw_networkx_labels(G_trace, pos, labels=node_labels, font_size=10)

plt.title(f"Witness Graph: {names[adam_idx]} to {names[abram_idx]}")
plt.show()

AttributeError: 'Provenance' object has no attribute 'GetWitnessGraph'